# 25 — Enhanced Feature Engineering

Adds features motivated directly by EDA findings:

1. **Collaboration scope** — ordinal (Domestic=1, Regional=2, Global=3)
2. **Author count transformations** — log-transform + 7-bucket encoding
3. **Open Access category encoding** — treat Unknown as explicit level
4. **ASJC field encoding** — frequency rank + training-set target encoding
5. **Abstract length** — word count (log-transformed)
6. **Field-normalized target** — z-score within primary ASJC field

**Input:** `data/processed/merged_data.pkl`  
**Output:** `data/features/enhanced_features.pkl`, `data/features/y_field_normalized.pkl`

In [ ]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', None)

PROC_DIR = Path('../data/processed')
FEAT_DIR = Path('../data/features')
FEAT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_pickle(PROC_DIR / 'merged_data.pkl')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

## 1. Collaboration Scope (Ordinal)

EDA showed a clean monotonic relationship:
- Domestic (1 country): median citations ~9
- Regional (2-3 countries): median citations ~13
- Global (4+ countries): median citations ~20

Encode as ordinal 1/2/3 so models can exploit the ordering directly.

In [ ]:
enhanced = pd.DataFrame(index=df.index)

n_countries = df['Number of Countries/Regions'].fillna(1)

enhanced['collab_scope'] = np.select(
    [n_countries == 1, n_countries <= 3],
    [1, 2],
    default=3
).astype(int)

# Also keep dummy versions for linear models
enhanced['collab_domestic'] = (enhanced['collab_scope'] == 1).astype(int)
enhanced['collab_regional'] = (enhanced['collab_scope'] == 2).astype(int)
enhanced['collab_global']   = (enhanced['collab_scope'] == 3).astype(int)

print('Collaboration scope distribution:')
print(enhanced['collab_scope'].value_counts().sort_index())
print(f'\nDistribution (%):')
print((enhanced['collab_scope'].value_counts(normalize=True).sort_index() * 100).round(1))

## 2. Author Count Transformations

EDA showed:
- Distribution is extremely right-skewed (outliers to 350+)
- Monotonic citation lift across buckets (1 author: median 3, 50+: median 37)

Create: log-transform, capped raw count, and 7-bucket ordinal.

In [ ]:
n_authors = df['Number of Authors'].fillna(df['Number of Authors'].median())

# Log-transform (handles skew; the EDA monotonic trend survives log scale)
enhanced['log_num_authors'] = np.log1p(n_authors)

# Cap at 99th percentile then keep raw (for tree models)
cap_99 = n_authors.quantile(0.99)
enhanced['num_authors_capped'] = n_authors.clip(upper=cap_99)

# 7-bucket ordinal matching the EDA plot
enhanced['author_bucket'] = pd.cut(
    n_authors,
    bins=[0, 1, 3, 6, 10, 20, 50, np.inf],
    labels=[1, 2, 3, 4, 5, 6, 7]
).astype(int)

print('Author count stats:')
print(n_authors.describe().round(1))
print(f'\n99th percentile cap: {cap_99:.0f}')
print('\nBucket distribution:')
bucket_labels = ['1', '2-3', '4-6', '7-10', '11-20', '21-50', '50+']
print(enhanced['author_bucket'].value_counts().sort_index()
      .rename(dict(enumerate(bucket_labels, 1))))

## 3. Open Access Category Encoding

EDA showed ~55% of papers have NaN OA status — current code collapses this to binary,
losing information about which OA type. Fix: treat each category (including Unknown) as its
own level via one-hot encoding.

In [ ]:
# Determine the OA column name (handle slight naming variation)
oa_col = 'Open access' if 'Open access' in df.columns else 'Open Access'

oa = df[oa_col].fillna('Unknown')

print('OA category distribution:')
print(oa.value_counts())

# One-hot encode all categories (Unknown gets its own column)
oa_dummies = pd.get_dummies(oa, prefix='oa').astype(int)

# Explicit unknown flag for models that don't see dummies
enhanced['oa_is_unknown'] = (oa == 'Unknown').astype(int)
enhanced['oa_has_gold']   = oa.str.contains('Gold', na=False).astype(int)
enhanced['oa_has_green']  = oa.str.contains('Green', na=False).astype(int)

# Concat OA dummies
enhanced = pd.concat([enhanced, oa_dummies], axis=1)

print(f'\nOA features added: {oa_dummies.shape[1] + 3}')

## 4. ASJC Field Encoding

High cardinality (many unique values) — use two encodings:
- **Frequency rank**: rank by paper count (1 = most common field), normalized 0-1
- **Target encoding**: mean log-citations per field, computed on training set only (2015-2017)
  to prevent leakage

Both are computed from `primary_asjc` (first listed field per paper).

In [ ]:
asjc_col = 'All Science Journal Classification (ASJC) field name'

# Primary field = first listed
primary_asjc = df[asjc_col].str.split(';').str[0].str.strip().fillna('Unknown')

# ---- Frequency rank (no leakage: computed on full dataset, only reflects corpus structure) ----
field_counts = primary_asjc.value_counts()
n_fields = len(field_counts)
# Rank 1 = most common; normalize to [0, 1]
field_rank = {field: (n_fields - rank) / n_fields
              for rank, field in enumerate(field_counts.index)}

enhanced['asjc_freq_rank'] = primary_asjc.map(field_rank).fillna(0)

# ---- Target encoding (mean log-citations per field, training years only) ----
train_mask = df['Year'].isin([2015, 2016, 2017])
train_df   = df[train_mask].copy()
train_df['primary_asjc'] = primary_asjc[train_mask]
train_df['log_citations'] = np.log1p(train_df['Citations'])

global_mean = train_df['log_citations'].mean()

# Smoothed target encoding: blend field mean with global mean weighted by field count
# smooth = k / (k + count), where k=10 controls how much we trust small samples
k = 10
field_stats = train_df.groupby('primary_asjc')['log_citations'].agg(['mean', 'count'])
field_stats['smoothed_mean'] = (
    (field_stats['count'] * field_stats['mean'] + k * global_mean)
    / (field_stats['count'] + k)
)

field_target_map = field_stats['smoothed_mean'].to_dict()

enhanced['asjc_target_enc'] = primary_asjc.map(field_target_map).fillna(global_mean)

# Also store primary_asjc for field-normalization step below
enhanced['primary_asjc_raw'] = primary_asjc  # keep for normalization, drop before modeling

print(f'Unique primary ASJC fields: {primary_asjc.nunique()}')
print(f'\nTop 10 fields by paper count:')
print(field_counts.head(10).to_string())
print(f'\nTarget encoding sample (top 5 by smoothed mean log-citations):')
print(field_stats.nlargest(5, 'smoothed_mean')[['mean', 'count', 'smoothed_mean']].round(3))

## 5. Abstract Length

Word count of abstract, log-transformed. Longer abstracts tend to describe more complete work.

In [ ]:
abstract_col = 'Abstract' if 'Abstract' in df.columns else None

if abstract_col:
    word_count = df[abstract_col].fillna('').str.split().str.len()
    enhanced['abstract_len_log'] = np.log1p(word_count)
    enhanced['abstract_missing'] = df[abstract_col].isna().astype(int)
    print('Abstract word count statistics:')
    print(word_count.describe().round(1))
    print(f'Missing abstracts: {df[abstract_col].isna().sum()}')
else:
    enhanced['abstract_len_log'] = 0
    enhanced['abstract_missing'] = 1
    print('Abstract column not found — filling with 0')

## 6. Field-Normalized Target

Within each primary ASJC field, compute z-score of log-citations:

    z = (log1p(citations) - field_mean) / field_std

Field mean and std are computed from **training papers only** (2015-2017) to prevent leakage.
Test papers use the training set's field statistics.

This target is better for regression because it removes field-level citation norm differences
(EDA showed 3x variation in median citations across fields).

In [ ]:
log_citations = np.log1p(df['Citations'])

# Compute field stats on training set only
train_primary_asjc = primary_asjc[train_mask]
train_log_cit = log_citations[train_mask]

field_norm_stats = pd.DataFrame({
    'field_mean': train_log_cit.groupby(train_primary_asjc).mean(),
    'field_std':  train_log_cit.groupby(train_primary_asjc).std().fillna(1),
})

# Global fallback for fields not in training set
global_std = train_log_cit.std()

field_mean_map = field_norm_stats['field_mean'].to_dict()
field_std_map  = field_norm_stats['field_std'].to_dict()

paper_field_mean = primary_asjc.map(field_mean_map).fillna(global_mean)
paper_field_std  = primary_asjc.map(field_std_map).fillna(global_std)
paper_field_std  = paper_field_std.clip(lower=0.01)  # avoid division by zero

y_field_normalized = (log_citations - paper_field_mean) / paper_field_std
y_field_normalized.name = 'y_field_normalized'

print('Field-normalized target statistics (all papers):')
print(y_field_normalized.describe().round(3))
print(f'\nTrain set (2015-2017):')
print(y_field_normalized[train_mask].describe().round(3))
print(f'\nTest set (2018-2020):')
test_mask = df['Year'].isin([2018, 2019, 2020])
print(y_field_normalized[test_mask].describe().round(3))

## 7. Validate: No Data Leakage

All features above must be computable at publication time:
- Collaboration scope ✅ (from author affiliations on paper)
- Author count transformations ✅ (from byline)
- OA category ✅ (determined at publication)
- ASJC frequency rank ✅ (reflects journal classification, static)
- ASJC target encoding ✅ (computed from training years only)
- Abstract length ✅ (abstract written before publication)
- Field-normalized target ✅ (uses training-set field stats only)

In [ ]:
# Drop the raw string column before saving (not a numeric feature)
feature_cols = [c for c in enhanced.columns if c != 'primary_asjc_raw']
enhanced_numeric = enhanced[feature_cols]

print('Enhanced features shape:', enhanced_numeric.shape)
print('\nFeature list:')
for col in enhanced_numeric.columns:
    print(f'  {col}: {enhanced_numeric[col].dtype}, '
          f'missing={enhanced_numeric[col].isna().sum()}')

print('\nAny NaN in enhanced features?', enhanced_numeric.isna().any().any())

# Sanity: target encoding only uses training-year papers
# (if we used test-set citations to compute field_target_map, that would be leakage)
print(f'\nTarget encoding computed from {train_mask.sum()} training papers (2015-2017) only ✅')

## 8. Save Outputs

In [ ]:
enhanced_numeric.to_pickle(FEAT_DIR / 'enhanced_features.pkl')
y_field_normalized.to_pickle(FEAT_DIR / 'y_field_normalized.pkl')

# Also save field norm stats so inference can apply same normalization
field_norm_stats.to_pickle(FEAT_DIR / 'field_norm_stats.pkl')

# Save ASJC target encoding map
import pickle
with open(FEAT_DIR / 'asjc_target_enc_map.pkl', 'wb') as f:
    pickle.dump({'field_target_map': field_target_map,
                 'global_mean': global_mean,
                 'field_norm_stats': field_norm_stats.to_dict()}, f)

print('Saved:')
print(f'  data/features/enhanced_features.pkl  — {enhanced_numeric.shape}')
print(f'  data/features/y_field_normalized.pkl — {y_field_normalized.shape}')
print(f'  data/features/field_norm_stats.pkl')
print(f'  data/features/asjc_target_enc_map.pkl')

## 9. How to Use in Notebook 23 (Final Combination)

Add to `23_feature_engineering_final.ipynb`:

```python
# Load enhanced features
enhanced_features = pd.read_pickle(feature_dir / 'enhanced_features.pkl')

# Combine with existing
X = pd.concat([text_features, venue_features, author_features,
               additional_features, enhanced_features], axis=1)

# Load field-normalized target for regression experiments
y_field_norm = pd.read_pickle(feature_dir / 'y_field_normalized.pkl')
```

**New features added:**

| Feature | Type | Rationale |
|---|---|---|
| `collab_scope` | Ordinal 1-3 | 2x citation lift domestic→global |
| `collab_domestic/regional/global` | Binary dummies | For linear models |
| `log_num_authors` | Continuous | Handles right-skew, monotonic signal |
| `num_authors_capped` | Continuous | Capped at 99th pct for trees |
| `author_bucket` | Ordinal 1-7 | Matches EDA bucket analysis |
| `oa_is_unknown` | Binary | Explicit missing indicator |
| `oa_has_gold` / `oa_has_green` | Binary | OA type signals |
| `oa_*` dummies | Binary | Full OA category encoding |
| `asjc_freq_rank` | 0-1 | Field commonness in corpus |
| `asjc_target_enc` | Continuous | Smoothed mean log-cit per field (train only) |
| `abstract_len_log` | Continuous | Paper completeness proxy |
| `abstract_missing` | Binary | Missing abstract indicator |

**New target:**
- `y_field_normalized`: z-score within ASJC field — removes field-level citation norm differences

In [ ]:
print('=' * 60)
print('ENHANCED FEATURE ENGINEERING SUMMARY')
print('=' * 60)
print(f'Total papers: {len(enhanced_numeric):,}')
print(f'Enhanced features: {enhanced_numeric.shape[1]}')
print()
print('Feature breakdown:')
print(f'  Collaboration scope : 4 features (ordinal + 3 dummies)')
oa_cols = [c for c in enhanced_numeric.columns if c.startswith('oa_')]
print(f'  Author count        : 3 features (log, capped, bucket)')
print(f'  Open access         : {len(oa_cols)} features (dummies + flags)')
print(f'  ASJC encoding       : 2 features (freq rank, target enc)')
print(f'  Abstract length     : 2 features (log word count, missing flag)')
print()
print('Targets:')
print(f'  y_field_normalized  : z-score within ASJC field')
print(f'  (existing)          : y_regression_log, y_classification')
print()
print('All features are ex-ante (computable at publication time) ✅')